# Capstone EDA — EPA AQS Air Quality + CDPH Respiratory Health Data
**Project: Air Quality Prediction and Respiratory Health Impact Modeling in California (2020–2024)**

Team: Vinh, Paola Rodriguez, Allison Mckernan | Course: ADS-599 Capstone

---

### Datasets Used
| # | Dataset | Source | Granularity | Coverage |
|---|---------|--------|-------------|----------|
| 1 | EPA AQS Daily AQI by County | EPA AQS | County / Daily | 2020–2024 |
| 2 | Asthma Hospitalization Rates | CDPH / CHHS | County / Annual | 2020–2023 |
| 3 | Asthma ED Visit Rates | CDPH / CHHS | County / Annual | 2020–2023 |

### Notebook Structure
- **Part A** — EPA AQS EDA (Air Quality)
- **Part B** — CDPH Health Data EDA (Hospitalizations + ED Visits)
- **Part C** — Joint Analysis (AQI vs Health Outcomes by County)

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120

# ── Set your data paths here ──────────────────────────────────────────────────
AQS_DIR   = Path('./data/aqs')          # folder with daily_aqi_by_county_YYYY.csv files
HOSP_CSV  = Path('./data/asthma_hosp_by_county.csv')   # CDPH hospitalization CSV
ED_CSV    = Path('./data/asthma_ed_by_county.csv')     # CDPH ED visits CSV
# ─────────────────────────────────────────────────────────────────────────────

print('Libraries loaded OK.')

In [ ]:
# ==========================================================
# Import utility functions from src/ modules
# ==========================================================

import sys
sys.path.insert(0, ".")

from src.data_preprocessing import (
    load_aqs_data,
    filter_california,
    add_population_weights,
    compute_weekly_state_aqi,
    compute_annual_county_aqi,
    filter_main,
)
from src.feature_engineering import add_grouped_lag, encode_county
from src.visualization import plot_weekly_aqi

print("src modules loaded successfully.")

---
# PART A — EPA AQS Air Quality EDA

### A1. Download Instructions
Go to: **https://aqs.epa.gov/aqsweb/airdata/download_files.html**

Under *Daily Summary Data* → download `daily_aqi_by_county_YYYY.zip` for 2020, 2021, 2022, 2023, 2024.
Unzip all files into `./data/aqs/`

### A2. Load & Filter to California

In [ ]:
years = [2020, 2021, 2022, 2023, 2024]
frames = []

for yr in years:
    fp = AQS_DIR / f'daily_aqi_by_county_{yr}.csv'
    if fp.exists():
        df_yr = pd.read_csv(fp)
        frames.append(df_yr)
        print(f'{yr}: {len(df_yr):,} rows loaded')
    else:
        print(f'WARNING: {fp} not found — skipping')

df_aqs_raw = pd.concat(frames, ignore_index=True)
df_aqs = df_aqs_raw[df_aqs_raw['State Name'] == 'California'].copy()

df_aqs['Date']  = pd.to_datetime(df_aqs['Date'])
df_aqs['Year']  = df_aqs['Date'].dt.year
df_aqs['Month'] = df_aqs['Date'].dt.month

print(f'\nAll states: {len(df_aqs_raw):,} rows')
print(f'California: {len(df_aqs):,} rows')
print(f'Date range: {df_aqs["Date"].min().date()} → {df_aqs["Date"].max().date()}')

### A3. Initial Inspection

In [ ]:
print('Columns:', df_aqs.columns.tolist())
df_aqs.head()

In [ ]:
df_aqs.describe()

### A4. Missing Value Analysis

In [ ]:
missing = df_aqs.isnull().mean().mul(100).round(2).sort_values(ascending=False)
missing_df = missing[missing > 0].rename('Missing %')

if len(missing_df) > 0:
    missing_df.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.title('AQS — Missing Data (%) by Column')
    plt.xlabel('% Missing')
    plt.tight_layout()
    plt.show()
    print(missing_df)
else:
    print('No missing values in AQS data.')

### A5. County Coverage Heatmap

In [ ]:
county_col = 'county Name'  # exact column name from EPA AQS files

county_year = (
    df_aqs.groupby([county_col, 'Year'])
    .size()
    .unstack(fill_value=0)
)

print(f'Counties with data: {len(county_year)}')

fig, ax = plt.subplots(figsize=(10, 16))
sns.heatmap(
    county_year, annot=True, fmt='d',
    cmap='YlGnBu', linewidths=0.4, ax=ax,
    cbar_kws={'label': 'Days of Data'}
)
ax.set_title('County × Year — Days of AQI Data Available (CA 2020–2024)', fontsize=13)
ax.set_ylabel('County')
plt.tight_layout()
plt.show()

### A6. AQI Distribution & Category Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_aqs['AQI'].dropna(), bins=60, color='steelblue', edgecolor='white')
for val, color, label in [(50,'green','Good (50)'), (100,'orange','Moderate (100)'), (150,'red','Unhealthy (150)')]:
    axes[0].axvline(val, color=color, linestyle='--', linewidth=1, label=label)
axes[0].set_title('AQI Distribution — California 2020–2024')
axes[0].set_xlabel('AQI')
axes[0].legend(fontsize=8)

df_aqs['AQI'].plot(kind='box', ax=axes[1], patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('AQI Boxplot')
axes[1].set_ylabel('AQI')
plt.tight_layout()
plt.show()

In [ ]:
# AQI category distribution
cat_order  = ['Good','Moderate','Unhealthy for Sensitive Groups','Unhealthy','Very Unhealthy','Hazardous']
cat_colors = ['#00e400','#ffff00','#ff7e00','#ff0000','#8f3f97','#7e0023']

if 'Category' in df_aqs.columns:
    cat_counts = df_aqs['Category'].value_counts().reindex(
        [c for c in cat_order if c in df_aqs['Category'].unique()]
    )
    colors_used = [cat_colors[cat_order.index(c)] for c in cat_counts.index]
    cat_counts.plot(kind='bar', color=colors_used, edgecolor='white')
    plt.title('AQI Category Distribution — California 2020–2024')
    plt.ylabel('County-Days')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    pct = (cat_counts / cat_counts.sum() * 100).round(2)
    print(pd.DataFrame({'Count': cat_counts, '%': pct}))

### A7. Defining Pollutant Breakdown

In [ ]:
if 'Defining Parameter' in df_aqs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Overall
    df_aqs['Defining Parameter'].value_counts().plot(
        kind='bar', ax=axes[0], color='coral', edgecolor='white'
    )
    axes[0].set_title('Defining Pollutant — Overall')
    axes[0].set_ylabel('County-Days')
    axes[0].tick_params(axis='x', rotation=30)

    # By year (stacked %)
    poll_yr = df_aqs.groupby(['Year','Defining Parameter']).size().unstack(fill_value=0)
    poll_yr_pct = poll_yr.div(poll_yr.sum(axis=1), axis=0) * 100
    poll_yr_pct.plot(kind='bar', stacked=True, ax=axes[1],
                     colormap='tab10', edgecolor='white')
    axes[1].set_title('Defining Pollutant Share by Year')
    axes[1].set_ylabel('% of County-Days')
    axes[1].tick_params(axis='x', rotation=0)
    axes[1].legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)

    plt.suptitle('Which Pollutant Drives AQI — California 2020–2024', fontsize=12)
    plt.tight_layout()
    plt.show()

### A8. Daily Statewide Time Series

In [ ]:
daily_state = df_aqs.groupby('Date')['AQI'].mean().reset_index()
daily_state.columns = ['Date','Mean_AQI']

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_state['Date'], daily_state['Mean_AQI'],
        color='steelblue', linewidth=0.8, alpha=0.8)
for val, color, label in [
    (50,'green','Good (50)'),
    (100,'orange','Moderate (100)'),
    (150,'red','Unhealthy (150)')
]:
    ax.axhline(val, color=color, linestyle='--', linewidth=0.9, label=label)

# Wildfire season shading
wildfire_periods = [
    ('2020-08-15','2020-11-01','2020 Wildfire Season'),
    ('2021-07-01','2021-10-15','Dixie Fire 2021'),
    ('2023-06-01','2023-09-30','2023 Wildfire Season'),
]
for start, end, label in wildfire_periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.1, color='red', label=label)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.set_title('Daily Mean AQI — California 2020–2024 (All Counties Average)')
ax.set_ylabel('Mean AQI')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# AQI by year boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_aqs.boxplot(column='AQI', by='Year', ax=axes[0], patch_artist=True)
axes[0].set_title('AQI by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('AQI')
plt.sca(axes[0])
plt.title('AQI Distribution by Year')

# Monthly seasonality
monthly_avg = df_aqs.groupby('Month')['AQI'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(monthly_avg.index, monthly_avg.values, color='steelblue', edgecolor='white')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_labels)
axes[1].set_title('Seasonality — Average AQI by Month')
axes[1].set_ylabel('Mean AQI')

plt.suptitle('')
plt.tight_layout()
plt.show()

### A9. Top Polluted Counties — Individual Time Series

In [ ]:
top_counties = (
    df_aqs.groupby(county_col)['AQI']
    .mean()
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)
print('Top 6 most polluted counties (mean AQI):', top_counties)

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
axes = axes.flatten()

for i, county in enumerate(top_counties):
    s = df_aqs[df_aqs[county_col] == county].groupby('Date')['AQI'].mean()
    axes[i].plot(s.index, s.values, color='steelblue', linewidth=0.8)
    axes[i].axhline(100, color='orange', linestyle='--', linewidth=0.7)
    axes[i].axhline(150, color='red', linestyle='--', linewidth=0.7)
    axes[i].set_title(f'{county} County', fontsize=11)
    axes[i].set_ylabel('AQI')
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=40)

plt.suptitle('Daily AQI — Top 6 Most Polluted CA Counties (2020–2024)', fontsize=13)
plt.tight_layout()
plt.show()

### A10. Population-Weighted Annual & Weekly AQI by County

In [ ]:
ca_county_pop = {
    'Los Angeles':10014009,'San Diego':3298634,'Orange':3186989,
    'Riverside':2418185,'San Bernardino':2181654,'Santa Clara':1936259,
    'Alameda':1682353,'Sacramento':1585055,'Contra Costa':1153526,
    'Fresno':1008654,'Kern':909235,'San Francisco':873965,
    'Ventura':843843,'San Mateo':764442,'San Joaquin':779233,
    'Stanislaus':550660,'Sonoma':488863,'Tulare':473117,
    'Solano':447643,'Monterey':434061,'Santa Barbara':446475,
    'Placer':398329,'San Luis Obispo':282424,'Marin':258826,
    'Shasta':182155,'Merced':281202,'Butte':211632,
    'Yolo':220500,'El Dorado':192843,'Imperial':179702,
    'Kings':152940,'Madera':157327,'Napa':137744,
    'Humboldt':136310,'Nevada':103487,'Sutter':99633,
    'Mendocino':91305,'Yuba':83421,'Lake':68766,
    'Tehama':65084,'San Benito':62808,'Tuolumne':55810,
    'Calaveras':46221,'Siskiyou':43724,'Amador':41472,
    'Lassen':33159,'Del Norte':28650,'Glenn':28917,
    'Colusa':21547,'Plumas':20007,'Inyo':18970,
    'Mariposa':17203,'Trinity':12285,'Mono':14444,
    'Modoc':8661,'Sierra':3236,'Alpine':1204
}

pop_df = pd.DataFrame(list(ca_county_pop.items()), columns=[county_col,'Population'])
df_aqs_pop = df_aqs.merge(pop_df, on=county_col, how='left')

unmapped = df_aqs_pop[df_aqs_pop['Population'].isna()][county_col].unique()
print('Unmapped counties:', unmapped if len(unmapped) else 'None — all mapped!')

df_aqs_pop = df_aqs_pop.dropna(subset=['Population','AQI'])

In [ ]:
# Annual mean AQI by county (for joining with CDPH annual health data)
annual_county_aqi = (
    df_aqs_pop.groupby([county_col, 'Year'])['AQI']
    .mean()
    .reset_index()
    .rename(columns={'AQI':'Mean_AQI', county_col:'COUNTY'})
)
annual_county_aqi['COUNTY'] = annual_county_aqi['COUNTY'].str.upper()

print('Annual county AQI shape:', annual_county_aqi.shape)
annual_county_aqi.head(10)

In [ ]:
# Population-weighted weekly statewide AQI series (for AQI forecast model)
df_aqs_pop['WeekEnd'] = df_aqs_pop['Date'] + pd.to_timedelta(
    6 - df_aqs_pop['Date'].dt.dayofweek, unit='d'
)

weekly_state = (
    df_aqs_pop.groupby('WeekEnd')
    .apply(lambda g: np.average(g['AQI'], weights=g['Population']))
    .reset_index()
)
weekly_state.columns = ['WeekEnd','PopWeighted_AQI']
weekly_state = weekly_state.sort_values('WeekEnd')

fig, ax = plt.subplots(figsize=(16,5))
ax.plot(weekly_state['WeekEnd'], weekly_state['PopWeighted_AQI'],
        color='steelblue', linewidth=1.2, marker='o', markersize=2)
ax.axhline(50, color='green', linestyle='--', linewidth=0.9, label='Good (50)')
ax.axhline(100, color='orange', linestyle='--', linewidth=0.9, label='Moderate (100)')
ax.set_title('Population-Weighted Weekly Statewide AQI — California 2020–2024')
ax.set_ylabel('AQI (Pop-Weighted)')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nWeekly statewide AQI stats:')
print(weekly_state['PopWeighted_AQI'].describe().round(2))

In [ ]:
# Autocorrelation & rolling stationarity check
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

autocorrelation_plot(weekly_state['PopWeighted_AQI'].dropna(), ax=axes[0])
axes[0].set_title('Autocorrelation — Weekly Statewide AQI')
axes[0].set_xlim(0, 52)

roll_mean = weekly_state['PopWeighted_AQI'].rolling(4).mean()
roll_std  = weekly_state['PopWeighted_AQI'].rolling(4).std()
axes[1].plot(weekly_state['WeekEnd'], weekly_state['PopWeighted_AQI'],
             alpha=0.4, label='Weekly AQI', color='steelblue')
axes[1].plot(weekly_state['WeekEnd'], roll_mean, color='red', label='4-wk Rolling Mean')
axes[1].plot(weekly_state['WeekEnd'], roll_std, color='orange', label='4-wk Rolling Std')
axes[1].set_title('Rolling Mean & Std (Stationarity Check)')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### A11. Save AQS Outputs

In [ ]:
Path('./data/outputs').mkdir(parents=True, exist_ok=True)

weekly_state.to_csv('./data/outputs/weekly_statewide_aqi.csv', index=False)
print('Saved: weekly_statewide_aqi.csv')

annual_county_aqi.to_csv('./data/outputs/annual_county_aqi.csv', index=False)
print('Saved: annual_county_aqi.csv')

weekly_county = (
    df_aqs_pop.groupby(['WeekEnd', county_col])['AQI']
    .mean().reset_index()
    .rename(columns={'AQI':'Mean_AQI', county_col:'County'})
)
weekly_county.to_csv('./data/outputs/weekly_county_aqi.csv', index=False)
print('Saved: weekly_county_aqi.csv')

---
# PART B — CDPH Respiratory Health Data EDA

### Download Instructions

**Dataset 1 — Asthma Hospitalization Rates by County (2015–2023):**
https://data.chhs.ca.gov/dataset/asthma-hospitalization-rates-by-county
→ Click **Download** next to *Asthma Hospitalization Rates by County 2015-present (CSV)*
→ Save as `./data/asthma_hosp_by_county.csv`

**Dataset 2 — Asthma ED Visit Rates by County (2015–2023):**
https://data.chhs.ca.gov/dataset/asthma-emergency-department-visit-rates
→ Click **Download** next to *Asthma ED Visit Rates by County 2015-Present (CSV)*
→ Save as `./data/asthma_ed_by_county.csv`

### B1. Load Health Datasets

In [ ]:
df_hosp = pd.read_csv(HOSP_CSV, encoding='latin-1')
df_ed   = pd.read_csv(ED_CSV, encoding='latin-1')

print('Hospitalization dataset shape:', df_hosp.shape)
print('ED visits dataset shape:      ', df_ed.shape)

In [ ]:
print('--- Hospitalization Columns ---')
print(df_hosp.columns.tolist())
df_hosp.head()

In [ ]:
print('--- ED Visits Columns ---')
print(df_ed.columns.tolist())
df_ed.head()

### B2. Filter to Total Population, All Ages, 2020–2023

In [ ]:
# filter_main is imported from src/data_preprocessing.py
# It filters CDPH health data to Total population / All ages,
# years 2020-2023, county-level rows only, and cleans the count column.

hosp_clean = filter_main(df_hosp, "NUMBER OF HOSPITALIZATIONS")
ed_clean   = filter_main(df_ed,   "NUMBER OF ED VISITS")

print("Hospitalization rows (2020-2023, all ages, county-level):", len(hosp_clean))
print("ED visit rows (2020-2023, all ages, county-level):       ", len(ed_clean))
hosp_clean.head()

### B3. Missing Value Analysis

In [ ]:
for name, df in [('Hospitalization', hosp_clean), ('ED Visits', ed_clean)]:
    print(f'\n--- {name} Missing Values ---')
    m = df.isnull().mean().mul(100).round(2)
    print(m[m > 0] if (m > 0).any() else 'No missing values')

    # Suppressed rate check (small counties suppressed per CDPH guidelines)
    if 'COMMENT' in df.columns:
        suppressed = df[df['COMMENT'].notna() & (df['COMMENT'] != '')]
        print(f'  Suppressed/flagged rows: {len(suppressed)}')
        if len(suppressed) > 0:
            print(suppressed[['COUNTY','YEAR','COMMENT']].head(10))

### B4. Hospitalization & ED Visit Counts by Year (Statewide Trend)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, col, title, color in [
    (axes[0], hosp_clean, 'NUMBER OF HOSPITALIZATIONS',
     'Asthma Hospitalizations by Year — CA', 'steelblue'),
    (axes[1], ed_clean,   'NUMBER OF ED VISITS',
     'Asthma ED Visits by Year — CA', 'coral')
]:
    yr_total = df.groupby('YEAR')[col].sum()
    ax.bar(yr_total.index.astype(str), yr_total.values,
           color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.set_xlabel('Year')
    for i, (yr, val) in enumerate(yr_total.items()):
        ax.text(i, val + val*0.01, f'{int(val):,}', ha='center', fontsize=9)

plt.suptitle('CDPH Statewide Respiratory Health Trends (2020–2023)', fontsize=12)
plt.tight_layout()
plt.show()

### B5. County-Level Distribution — Hospitalizations

In [ ]:
# Total hospitalizations per county across 2020–2023
hosp_county = (
    hosp_clean.groupby('COUNTY')['NUMBER OF HOSPITALIZATIONS']
    .sum()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 counties
hosp_county.head(20).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 Counties — Total Asthma Hospitalizations (2020–2023)')
axes[0].set_ylabel('Total Hospitalizations')
axes[0].tick_params(axis='x', rotation=45)

# Rate heatmap by county & year
hosp_rate_pivot = hosp_clean.pivot_table(
    index='COUNTY', columns='YEAR',
    values='AGE-ADJUSTED HOSPITALIZATION RATE',
    aggfunc='first'
)
# Show top 20 by mean rate
hosp_rate_pivot['mean'] = hosp_rate_pivot.mean(axis=1)
top20_rate = hosp_rate_pivot.nlargest(20, 'mean').drop(columns='mean')

sns.heatmap(
    top20_rate.astype(float), ax=axes[1],
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, cbar_kws={'label': 'Rate per 10,000'}
)
axes[1].set_title('Age-Adjusted Hospitalization Rate by County & Year\n(Top 20 Counties)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

### B6. County-Level Distribution — ED Visits

In [ ]:
ed_county = (
    ed_clean.groupby('COUNTY')['NUMBER OF ED VISITS']
    .sum()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ed_county.head(20).plot(kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Top 20 Counties — Total Asthma ED Visits (2020–2023)')
axes[0].set_ylabel('Total ED Visits')
axes[0].tick_params(axis='x', rotation=45)

ed_rate_pivot = ed_clean.pivot_table(
    index='COUNTY', columns='YEAR',
    values='AGE-ADJUSTED ED VISIT RATE',
    aggfunc='first'
)
ed_rate_pivot['mean'] = ed_rate_pivot.mean(axis=1)
top20_ed = ed_rate_pivot.nlargest(20, 'mean').drop(columns='mean')

sns.heatmap(
    top20_ed.astype(float), ax=axes[1],
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, cbar_kws={'label': 'Rate per 10,000'}
)
axes[1].set_title('Age-Adjusted ED Visit Rate by County & Year\n(Top 20 Counties)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

### B7. Year-over-Year Trend per County (Top 10)

In [ ]:
top10_hosp = hosp_county.head(10).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for county in top10_hosp:
    s = hosp_clean[hosp_clean['COUNTY'] == county].set_index('YEAR')['NUMBER OF HOSPITALIZATIONS']
    axes[0].plot(s.index, s.values, marker='o', label=county)

axes[0].set_title('Asthma Hospitalizations by Year — Top 10 Counties')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Hospitalizations')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].set_xticks([2020,2021,2022,2023])

top10_ed_list = ed_county.head(10).index.tolist()
for county in top10_ed_list:
    s = ed_clean[ed_clean['COUNTY'] == county].set_index('YEAR')['NUMBER OF ED VISITS']
    axes[1].plot(s.index, s.values, marker='o', label=county)

axes[1].set_title('Asthma ED Visits by Year — Top 10 Counties')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('ED Visits')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].set_xticks([2020,2021,2022,2023])

plt.suptitle('Year-over-Year Respiratory Health Trends by County (2020–2023)', fontsize=12)
plt.tight_layout()
plt.show()

### B8. Save Health Outputs

In [ ]:
hosp_clean.to_csv('./data/outputs/asthma_hosp_clean.csv', index=False)
ed_clean.to_csv('./data/outputs/asthma_ed_clean.csv', index=False)
print('Saved: asthma_hosp_clean.csv')
print('Saved: asthma_ed_clean.csv')

---
# PART C — Joint Analysis: AQI vs Respiratory Health by County

This is the core of your lagged admissions model — merging AQI and health outcomes at the county level.

### C1. Merge AQI + Hospitalizations by County & Year

In [ ]:
# Standardize county names to title case for joining
annual_county_aqi['County_join'] = annual_county_aqi['COUNTY'].str.title()
hosp_clean['County_join'] = hosp_clean['COUNTY'].str.title()
ed_clean['County_join']   = ed_clean['COUNTY'].str.title()

# Merge AQI + hospitalizations
df_joint = annual_county_aqi.merge(
    hosp_clean[['County_join','YEAR','NUMBER OF HOSPITALIZATIONS',
                'AGE-ADJUSTED HOSPITALIZATION RATE']],
    left_on=['County_join','Year'],
    right_on=['County_join','YEAR'],
    how='inner'
).drop(columns='YEAR')

# Also merge ED visits
df_joint = df_joint.merge(
    ed_clean[['County_join','YEAR','NUMBER OF ED VISITS',
              'AGE-ADJUSTED ED VISIT RATE']],
    left_on=['County_join','Year'],
    right_on=['County_join','YEAR'],
    how='inner'
).drop(columns='YEAR')

print('Joint dataset shape:', df_joint.shape)
print('Years covered:', sorted(df_joint['Year'].unique()))
print('Counties matched:', df_joint['County_join'].nunique())
df_joint.head()

### C2. Correlation: Mean AQI vs Hospitalization Rate by County

In [ ]:
df_joint_clean = df_joint.dropna(subset=[
    'Mean_AQI','AGE-ADJUSTED HOSPITALIZATION RATE','AGE-ADJUSTED ED VISIT RATE'
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_col, color, title in [
    (axes[0], 'AGE-ADJUSTED HOSPITALIZATION RATE', 'steelblue',
     'Mean AQI vs Asthma Hosp Rate by County'),
    (axes[1], 'AGE-ADJUSTED ED VISIT RATE', 'coral',
     'Mean AQI vs Asthma ED Visit Rate by County')
]:
    ax.scatter(
        df_joint_clean['Mean_AQI'],
        df_joint_clean[y_col],
        alpha=0.5, color=color, edgecolors='white', s=50
    )
    # Trend line
    m, b = np.polyfit(
        df_joint_clean['Mean_AQI'].dropna(),
        df_joint_clean[y_col].dropna(), 1
    )
    x_line = np.linspace(df_joint_clean['Mean_AQI'].min(),
                         df_joint_clean['Mean_AQI'].max(), 100)
    ax.plot(x_line, m*x_line+b, 'k--', linewidth=1.2, label=f'Trend (slope={m:.2f})')
    r = df_joint_clean[['Mean_AQI', y_col]].corr().iloc[0,1]
    ax.set_title(f'{title}\nr = {r:.3f}')
    ax.set_xlabel('Mean Annual AQI')
    ax.set_ylabel('Age-Adjusted Rate per 10,000')
    ax.legend(fontsize=8)

plt.suptitle('AQI vs Respiratory Health Outcomes — County Level (2020–2023)', fontsize=12)
plt.tight_layout()
plt.show()

### C3. Correlation Matrix

In [ ]:
corr_cols = [
    'Mean_AQI',
    'AGE-ADJUSTED HOSPITALIZATION RATE',
    'NUMBER OF HOSPITALIZATIONS',
    'AGE-ADJUSTED ED VISIT RATE',
    'NUMBER OF ED VISITS'
]
corr_matrix = df_joint_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix — AQI & Respiratory Health Outcomes')
plt.tight_layout()
plt.show()

### C4. Year-Lagged Correlation (AQI → Health Outcomes with 1-Year Lag)

In [ ]:
# Create lagged AQI: does this year's AQI predict NEXT year's admissions?
aqi_lag = annual_county_aqi.copy()
aqi_lag['Year_next'] = aqi_lag['Year'] + 1

df_lag = hosp_clean[['County_join','YEAR','AGE-ADJUSTED HOSPITALIZATION RATE']].copy()
df_lag = df_lag.rename(columns={'YEAR':'Year'})

df_lagged = aqi_lag.merge(
    df_lag,
    left_on=['County_join','Year_next'],
    right_on=['County_join','Year'],
    how='inner',
    suffixes=('_aqi','_health')
).dropna(subset=['Mean_AQI','AGE-ADJUSTED HOSPITALIZATION RATE'])

r_lag = df_lagged[['Mean_AQI','AGE-ADJUSTED HOSPITALIZATION RATE']].corr().iloc[0,1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    df_lagged['Mean_AQI'],
    df_lagged['AGE-ADJUSTED HOSPITALIZATION RATE'],
    alpha=0.5, color='mediumpurple', edgecolors='white', s=50
)
m, b = np.polyfit(df_lagged['Mean_AQI'], df_lagged['AGE-ADJUSTED HOSPITALIZATION RATE'], 1)
x_line = np.linspace(df_lagged['Mean_AQI'].min(), df_lagged['Mean_AQI'].max(), 100)
ax.plot(x_line, m*x_line+b, 'k--', linewidth=1.2)
ax.set_title(f'1-Year Lagged: AQI (Year T) vs Hosp Rate (Year T+1)\nr = {r_lag:.3f}')
ax.set_xlabel('Mean AQI (Year T)')
ax.set_ylabel('Hosp Rate per 10,000 (Year T+1)')
plt.tight_layout()
plt.show()

print(f'Same-year correlation (r):    {df_joint_clean[["Mean_AQI","AGE-ADJUSTED HOSPITALIZATION RATE"]].corr().iloc[0,1]:.3f}')
print(f'1-year lagged correlation (r): {r_lag:.3f}')

### C5. Save Joint Dataset

In [ ]:
# C5 — Save Joint Dataset (with re-merge fix applied first)

# Step 1: Clean whitespace and re-merge to fix county name mismatches
aqi_fixed  = annual_county_aqi.copy()
hosp_fixed = hosp_clean.copy()
ed_fixed   = ed_clean.copy()

for df in [aqi_fixed, hosp_fixed, ed_fixed]:
    df["County_join"] = df["County_join"].str.strip()

df_joint2 = aqi_fixed.merge(
    hosp_fixed[["County_join","YEAR","NUMBER OF HOSPITALIZATIONS","AGE-ADJUSTED HOSPITALIZATION RATE"]],
    left_on=["County_join","Year"], right_on=["County_join","YEAR"], how="inner"
).drop(columns="YEAR")

df_joint2 = df_joint2.merge(
    ed_fixed[["County_join","YEAR","NUMBER OF ED VISITS","AGE-ADJUSTED ED VISIT RATE"]],
    left_on=["County_join","Year"], right_on=["County_join","YEAR"], how="inner"
).drop(columns="YEAR")

print("Re-merge complete:", df_joint2.shape, "—", df_joint2['County_join'].nunique(), "counties")

# Step 2: Save
import os
os.makedirs('./data/outputs', exist_ok=True)
df_joint2.to_csv('./data/outputs/joint_aqi_health_county.csv', index=False)
print('Saved: joint_aqi_health_county.csv')
print('Shape:', df_joint2.shape)

print('\n=== JOINT DATASET SUMMARY ===')
print(df_joint2[['Mean_AQI','AGE-ADJUSTED HOSPITALIZATION RATE',
                       'AGE-ADJUSTED ED VISIT RATE']].describe().round(2))


### C6. County Match Diagnostics — Which Counties Dropped Out?

In [ ]:
# Counties in AQI data
aqi_counties = set(annual_county_aqi["County_join"].unique())

# Counties in health data
hosp_counties = set(hosp_clean["County_join"].unique())
ed_counties   = set(ed_clean["County_join"].unique())

# Counties in joint (matched)
joint_counties = set(df_joint_clean["County_join"].unique())

print("AQI counties      :", len(aqi_counties))
print("CDPH hosp counties:", len(hosp_counties))
print("CDPH ED counties  :", len(ed_counties))
print("Matched counties  :", len(joint_counties))

# AQI counties missing from CDPH
aqi_only = sorted(aqi_counties - hosp_counties)
print("\nIn AQI but NOT in CDPH hosp (" + str(len(aqi_only)) + "):")
for c in aqi_only:
    print("  " + c)

# CDPH counties missing from AQI
hosp_only = sorted(hosp_counties - aqi_counties)
print("\nIn CDPH hosp but NOT in AQI (" + str(len(hosp_only)) + "):")
for c in hosp_only:
    print("  " + c)

In [ ]:
# Build a name correction map to fix mismatches
name_fix = {
    # Add any mismatches found above here, e.g.:
    # "San Bernardino": "San Bernardino",  # example placeholder
}

# Check for near-matches using simple string overlap
unmatched = aqi_only
print("Possible name mismatches (AQI name -> closest CDPH name):")
for aqi_name in unmatched:
    candidates = [h for h in hosp_counties if aqi_name[:4].lower() in h.lower() or h[:4].lower() in aqi_name.lower()]
    if candidates:
        print(f"  AQI: '{aqi_name}'  ->  CDPH candidates: {candidates}")
    else:
        print(f"  AQI: '{aqi_name}'  ->  no close match found (likely suppressed small county)")

In [ ]:
# C6. County Match Diagnostics — Verify re-merge results
# Note: df_joint2 was already created in C5 above

aqi_counties  = set(annual_county_aqi["County_join"].unique())
hosp_counties = set(hosp_clean["County_join"].unique())
ed_counties   = set(ed_clean["County_join"].unique())
joint_counties = set(df_joint2["County_join"].unique())

print("AQI counties      :", len(aqi_counties))
print("CDPH hosp counties:", len(hosp_counties))
print("CDPH ED counties  :", len(ed_counties))
print("Matched counties  :", len(joint_counties))

still_missing = sorted(aqi_counties - set(df_joint2["County_join"].unique()))
print("\nStill unmatched (" + str(len(still_missing)) + ") - suppressed/no health data:")
for c in still_missing:
    print("  " + c)


---
## Final Summary

In [ ]:
print('='*65)
print('FULL EDA SUMMARY — Capstone ADS-599')
print('='*65)
print(f"AQS date range     : {df_aqs['Date'].min().date()} → {df_aqs['Date'].max().date()}")
print(f"AQS records (CA)   : {len(df_aqs):,} county-days")
print(f"AQS counties       : {df_aqs[county_col].nunique()}")
print(f"AQI mean / median  : {df_aqs['AQI'].mean():.1f} / {df_aqs['AQI'].median():.1f}")
print(f"Weekly AQI series  : {len(weekly_state)} weeks")
print()
print(f"CDPH Hosp rows     : {len(hosp_clean):,} (county-years 2020–2023)")
print(f"CDPH ED rows       : {len(ed_clean):,} (county-years 2020–2023)")
print(f"Counties in health : {hosp_clean['COUNTY'].nunique()}")
print()
print(f"Joint (merged) rows: {len(df_joint2):,}")
print(f"Counties matched   : {df_joint2['County_join'].nunique()}")
print()
print('Outputs saved to ./data/outputs/')
print('  weekly_statewide_aqi.csv   → AQI forecast model input')
print('  annual_county_aqi.csv      → county AQI for lag model')
print('  weekly_county_aqi.csv      → weekly county AQI')
print('  asthma_hosp_clean.csv      → cleaned hospitalization data')
print('  asthma_ed_clean.csv        → cleaned ED visit data')
print('  joint_aqi_health_county.csv → merged for modeling')
print('='*65)